# Preprocessing with scLucid

This notebook demonstrates the complete preprocessing workflow — from QC-filtered data to an analysis-ready AnnData with PCA, neighbors graph, and UMAP embedding.

## 1. Runtime Setup

In [ ]:
from scLucid.runtime import setup_runtime_environment

setup_runtime_environment()

In [ ]:
import matplotlib
matplotlib.use("Agg", force=True)

import scanpy as sc
import scLucid
from scLucid.preprocess import (
    WorkflowConfig,
    HVGConfig,
    NormalizationConfig,
    ScalingConfig,
    GraphConfig,
    IntegrationConfig,
    run_preprocessing,
    find_hvgs,
    select_hvg_sets,
    suggest_hvg_choice,
    evaluate_hvg_stability,
    plot_hvg_metrics,
    normalize_data,
    scale_data,
    batch_correction,
)

sc.settings.set_figure_params(dpi=100, facecolor="white")

## 2. Load QC-Filtered Data

In [ ]:
adata = sc.read_h5ad("data/pbmc3k.h5ad")
print(f"Cells: {adata.n_obs}, Genes: {adata.n_vars}")
print(f"Layers: {list(adata.layers.keys())}")
print(f".obs columns: {list(adata.obs.columns)[:8]}...")

## 3. Intelligent Preprocessing Recommendation

scLucid can analyze data characteristics and recommend preprocessing parameters before you run the full workflow.

In [ ]:
from scLucid.preprocess.intelligent import (
    IntelligentPreprocessRecommender,
    IntelligentPreprocessConfig,
)

recommender = IntelligentPreprocessRecommender(
    config=IntelligentPreprocessConfig()
)
strategy = recommender.recommend(adata, tissue_type="normal")

print(f"Data quality score: {strategy.data_profile.data_quality_score:.0f}/100")
print(f"Strategy: {strategy.data_profile.strategy_type}")
print(f"Recommended HVGs: {strategy.hvg.n_top_genes} (CI: {strategy.hvg.ci_lower}–{strategy.hvg.ci_upper})")
print(f"Recommended PCs: {strategy.pca.n_pcs} (CI: {strategy.pca.ci_lower}–{strategy.pca.ci_upper})")
print(f"Recommended neighbors: {strategy.neighbors.n_neighbors}")
if strategy.batch_correction:
    print(f"Batch correction: {strategy.batch_correction.recommended_method}")
else:
    print("Batch correction: not needed")

## 4. Normalization

Normalize to a target sum (default 10,000) and log1p-transform. scLucid stores the result in `layers['normalized']`.

In [ ]:
norm_config = NormalizationConfig(target_sum=1e4, output_layer="normalized")
adata = normalize_data(adata, config=norm_config)

print(f"Normalized layer shape: {adata.layers['normalized'].shape}")
print(f"Mean counts per cell after norm: {adata.layers['normalized'].sum(axis=1).mean():.0f}")

## 5. Highly Variable Gene (HVG) Selection

scLucid provides multiple HVG selection methods and diagnostics.

### 5a. Standard scanpy-based HVG

In [ ]:
hvg_config = HVGConfig(
    method="scanpy",
    flavor="seurat",
    n_top_genes=int(strategy.hvg.n_top_genes),  # Use recommendation
    exclude_gene_types=["mitochondrial", "ribosomal", "hemoglobin"],
)

adata = find_hvgs(adata, config=hvg_config, force=True, input_layer="normalized")

n_hvg = adata.var["highly_variable_scanpy_seurat"].sum()
print(f"Selected {n_hvg} HVGs out of {adata.n_vars}")

### 5b. Compare Multiple HVG Methods

In [ ]:
# Also run seurat_v3 for comparison
adata = find_hvgs(
    adata,
    config=HVGConfig(method="scanpy", flavor="seurat_v3", n_top_genes=2000),
    force=True,
    input_layer="normalized",
)

# Print overlap analysis
hvg_keys = ["highly_variable_scanpy_seurat", "highly_variable_scanpy_seurat_v3"]
suggest_hvg_choice(adata, hvg_keys=hvg_keys, mode="standard")

# Visualize dispersion vs mean
plot_hvg_metrics(adata, hvg_key="highly_variable_scanpy_seurat", show_gene_labels=False)

### 5c. HVG Stability Evaluation

Bootstrap resampling to assess how stable the HVG selection is.

In [ ]:
# Quick stability check (use n_bootstrap=20 for real analysis)
stability_result = evaluate_hvg_stability(
    adata,
    hvg_key="highly_variable_scanpy_seurat",
    n_bootstrap=10,
    sample_fraction=0.8,
    plot=True,
)

stability_score = stability_result.uns.get("sclucid", {}).get("preprocess", {}).get("hvg_stability", {}).get("overall_score", None)
print(f"HVG stability score: {stability_score:.3f}" if stability_score is not None else "Check .uns for stability info")

## 6. Scaling

Z-score scaling (zero-centered, unit variance) with max value clipping.

In [ ]:
scale_config = ScalingConfig(scale_zero_center=True, max_value=10)
adata = scale_data(adata, config=scale_config)

print(f"Scaled data mean: {adata.X.mean():.3f}")
print(f"Scaled data max: {adata.X.max():.3f}") 

## 7. PCA

Principal component analysis using the recommended number of PCs.

In [ ]:
n_pcs = int(strategy.pca.n_pcs)
sc.tl.pca(adata, n_comps=n_pcs, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=n_pcs, log=True, save="_variance.png")
print(f"Computed {n_pcs} PCs")

## 8. Neighbors Graph & UMAP

Build the k-NN graph and compute UMAP embedding.

In [ ]:
n_neighbors = int(strategy.neighbors.n_neighbors)
sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
sc.tl.umap(adata)

sc.pl.umap(adata, color=["sampleID"] if "sampleID" in adata.obs else None,
           title="UMAP — Preprocessed Data", save="_preprocess.png")

## 9. Batch Correction (Optional)

When batch effects are present, apply integration.

In [ ]:
# Check if batch correction is needed
if adata.obs.get("sampleID", pd.Series()).nunique() > 1:
    print(f"Multiple samples detected: {adata.obs['sampleID'].nunique()}")
    
    # Apply Harmony integration
    int_config = IntegrationConfig(method="harmony", batch_key="sampleID")
    adata = batch_correction(adata, config=int_config)
    
    # Re-run neighbors and UMAP on corrected embedding
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep="X_pca_harmony")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color=["sampleID"], title="UMAP — After Harmony", save="_harmony.png")
else:
    print("Single sample — batch correction skipped.")

## 10. One-Shot Workflow (Alternative)

For production use, the `run_preprocessing` function runs the entire pipeline in one call with checkpoint/resume support.

In [ ]:
# Build a complete workflow config using recommendations
workflow_config = WorkflowConfig(
    normalization=NormalizationConfig(target_sum=1e4, output_layer="normalized"),
    hvg=HVGConfig(
        method="scanpy",
        flavor="seurat",
        n_top_genes=int(strategy.hvg.n_top_genes),
        exclude_gene_types=["mitochondrial", "ribosomal"],
    ),
    scaling=ScalingConfig(scale_zero_center=True, max_value=10),
    graph=GraphConfig(n_pcs=int(strategy.pca.n_pcs), n_neighbors=int(strategy.neighbors.n_neighbors)),
    integration=IntegrationConfig(method=None),
    run_integration=False,
    save_dir="outputs/preprocess",
    n_jobs=4,
)

# Run the complete workflow
# adata = run_preprocessing(adata, config=workflow_config)

## 11. Review Summary

scLucid stores a machine-readable review summary at `adata.uns['sclucid']['preprocess']`.

In [ ]:
import json

review = adata.uns.get("sclucid", {}).get("preprocess", {})
if review:
    print("Preprocessing review summary:")
    print(json.dumps({k: str(v)[:120] for k, v in review.items()}, indent=2))
else:
    print("Run run_preprocessing() to generate a review summary.")